# 🌊 OceanoIA — Módulo 1: CNN para Identificación de Especies Marinas
En este apartado se encuentra el flujo completo que se implementó para el desarrollo de la red neuronal convolucional (CNN) en la cual se clasifican especies marinas capturadas en los mares de Costa Rica.

In [2]:
# Librerias
import os
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.image import imread
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Activation
from tensorflow.keras.preprocessing import image
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping

import warnings
warnings.filterwarnings('ignore')
print("TensorFlow versión:", tf.__version__)

TensorFlow versión: 2.21.0


## 3. Preprocesamiento y División de Datos
En este apartadose hizo la división del dataset raw (las imágenes en crudo) en 80% para entrenamiento y el 20% para pruebas

In [1]:
# El script 'src/data_prep.py' realiza esta división de manera balanceada (800 train, 200 test por clase)
# Mapeos aplicados:
# - Gilt-Head Bream -> dorado
# - Hourse Mackerel -> atun_aleta_amarilla
# - Red Sea Bream -> pargo_mancha
# - Sea Bass -> corvina_reina
# - Trout -> marlin_pez_vela (Especie protegida)
# - Shrimp -> tortuga_marina (Especie protegida)
# - Black Sea Sprat -> tiburon_martillo (Especie en veda)
# - Red Mullet + Striped Red Mullet -> otros

print("Para iniciar la partición, ejecuta en terminal: python src/data_prep.py")

Para iniciar la partición, ejecuta en terminal: python src/data_prep.py


In [ ]:
## 4. Aumento de Datos y Generadores de Keras
train_path = '../data/processed/train'
test_path = '../data/processed/test'
image_shape = (128, 128, 3)
batch_size = 32

# Definición de transformaciones para el conjunto de entrenamiento
image_gen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.10,
    height_shift_range=0.10,
    rescale=1/255,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

# Para pruebas solo escalamos los píxeles
test_gen_base = ImageDataGenerator(rescale=1/255)

In [ ]:
# Visualizar un ejemplo de transformación
sample_img_path = os.path.join(train_path, 'dorado', os.listdir(os.path.join(train_path, 'dorado'))[0])
sample_img = imread(sample_img_path)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.imshow(sample_img)
plt.title('Imagen Original')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(image_gen.random_transform(sample_img))
plt.title('Imagen Aumentada (Transformación)')
plt.axis('off')

plt.show()

In [ ]:
# Generadores de flujo
train_image_generator = image_gen.flow_from_directory(
    train_path,
    target_size=image_shape[:2],
    color_mode='rgb',
    batch_size=batch_size,
    class_mode='categorical'
)

test_image_generator = test_gen_base.flow_from_directory(
    test_path,
    target_size=image_shape[:2],
    color_mode='rgb',
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)

## 5. Diseño de la Arquitectura CNN

Implementamos la estructura sugerida en el diseño del proyecto para clasificación multiclase.

In [ ]:
model = Sequential()

# Bloque 1: Conv2D(32) -> MaxPooling
model.add(Conv2D(filters=32, kernel_size=(3,3), input_shape=image_shape, activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))

# Bloque 2: Conv2D(64) -> MaxPooling
model.add(Conv2D(filters=64, kernel_size=(3,3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))

# Bloque 3: Conv2D(128) -> MaxPooling
model.add(Conv2D(filters=128, kernel_size=(3,3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))

# Aplanado
model.add(Flatten())

# Capa Densa (256 Neuronas)
model.add(Dense(256, activation='relu'))

# Dropout para regularización
model.add(Dropout(0.5))

# Capa de Salida Softmax (8 clases)
model.add(Dense(8, activation='softmax'))

model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

## 6. Entrenamiento del Modelo

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

In [ ]:
# Entrenamiento
results = model.fit(
    train_image_generator,
    epochs=20,
    validation_data=test_image_generator,
    callbacks=[early_stop]
)

In [ ]:
# Guardar el modelo en formato Keras nativo
model.save('../models/cnn_especies.keras')

## 7. Evaluación del Modelo y Métricas

In [ ]:
# Graficar pérdidas
losses = pd.DataFrame(model.history.history)
losses[['loss', 'val_loss']].plot(figsize=(10, 4))
plt.title('Pérdida en Entrenamiento vs Validación')
plt.xlabel('Épocas')
plt.show()

In [ ]:
# Graficar precisión
losses[['accuracy', 'val_accuracy']].plot(figsize=(10, 4))
plt.title('Precisión (Accuracy) en Entrenamiento vs Validación')
plt.xlabel('Épocas')
plt.show()

In [ ]:
# Evaluación cuantitativa
model.evaluate(test_image_generator)

In [ ]:
# Reporte de Clasificación
preds = model.predict(test_image_generator)
predictions = np.argmax(preds, axis=-1)
true_classes = test_image_generator.classes
class_labels = list(test_image_generator.class_indices.keys())

print(classification_report(true_classes, predictions, target_names=class_labels))

In [ ]:
# Matriz de Confusión
cm = confusion_matrix(true_classes, predictions)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_labels, yticklabels=class_labels, cmap='Blues')
plt.title('Matriz de Confusión - CNN Especies')
plt.ylabel('Clase Real')
plt.xlabel('Predicción')
plt.show()

## 8. Predicción sobre una Nueva Imagen

In [ ]:
# Cargar una imagen aleatoria del test para validación
test_sub_dir = os.path.join(test_path, 'tortuga_marina')
test_img_name = os.listdir(test_sub_dir)[5]
ruta_nueva_imagen = os.path.join(test_sub_dir, test_img_name)

# Cargar y preprocesar para el modelo
nueva_imagen = image.load_img(ruta_nueva_imagen, target_size=image_shape[:2])
nueva_imagen_arr = image.img_to_array(nueva_imagen) / 255.0  # Normalización obligatoria
nueva_imagen_arr = np.expand_dims(nueva_imagen_arr, axis=0)

# Predicción
pred_probs = model.predict(nueva_imagen_arr)[0]
predicted_idx = np.argmax(pred_probs)
predicted_class = class_labels[predicted_idx]

plt.imshow(nueva_imagen)
plt.title(f"Predicción: {predicted_class} ({pred_probs[predicted_idx]*100:.2f}% de confianza)")
plt.axis('off')
plt.show()